##### Run the following code in virtual environment [PYTHON 3.11.9]

### Setup Dependencies

In [1]:
import sys
!{sys.executable} -m pip install langchain-community langchain-ollama langchain chromadb pypdf rapidfuzz

'c:\Users\neeha\Desktop\UTD' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
import sys
# This ensures we install exactly into the Python 3.11.9 kernel you are currently using
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install cryptography>=3.1 pycryptodome pypdf[crypto]

'c:\Users\neeha\Desktop\UTD' is not recognized as an internal or external command,
operable program or batch file.
'c:\Users\neeha\Desktop\UTD' is not recognized as an internal or external command,
operable program or batch file.


### Setup Paths

In [3]:
import os, glob

# These paths are relative to where your .ipynb file is saved
PDF_DIR = "./pdfs"
DB_DIR = "./chroma_db"

# Create the directory if it doesn't exist
if not os.path.exists(PDF_DIR):
    os.makedirs(PDF_DIR)
    print(f"Created {PDF_DIR} folder. Please move your PDFs there!")

pdfs = sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf")))

print("PDF_DIR:", PDF_DIR)
print("Found PDFs:", len(pdfs))
for p in pdfs:
    print("-", os.path.basename(p))

PDF_DIR: ./pdfs
Found PDFs: 10
- betrayal_official.pdf
- catan_official.pdf
- monopoly__deal_official.pdf
- monopoly_official.pdf
- pandemic_official.pdf
- risk_official.pdf
- root_official.pdf
- sushi_go_official.pdf
- ticket_to_ride_.pdf
- uno_official.pdf


### Functions & PDF Loading

In [ ]:
import os
import re
import glob
from langchain_community.document_loaders import PyPDFLoader

VALID_GAMES = {
    "uno", "monopoly", "catan", "monopoly_deal", "root", 
    "betrayal", "sushi_go", "pandemic", "ticket_to_ride", "risk"
}

def parse_game_from_filename(fname: str) -> str:
    base = os.path.splitext(fname)[0].lower()
    if base.endswith("_official"):
        base = base[:-len("_official")]
    base = re.sub(r"[^a-z0-9]+", "_", base)
    base = re.sub(r"_+", "_", base).strip("_")
    return base

pdf_paths = sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf")))
docs = []
for path in pdf_paths:
    fname = os.path.basename(path)
    game = parse_game_from_filename(fname)
    loaded = PyPDFLoader(path).load()
    for d in loaded:
        d.metadata.update({
            "game": game,
            "doc_type": "official_rules",
            "source_file": fname,
            "path": path
        })
    docs.extend(loaded)

print(f"Pages loaded: {len(docs)}")

c:\Users\neeha\Desktop\UTD S4\Agentic\The Game Codex\The Game Codex\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Chunking the Rulebooks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 800
CHUNK_OVERLAP = 300

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(docs)
print("Chunks created:", len(chunks))

### ChromaDB & Local Embeddings

In [ ]:
import os, shutil
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

# We use the Nomic-embed-text model you pulled for embeddings. You can use your own. 
# Make sure you have the Ollama server running locally with the model available before executing this part.
# Make sure the model used in future can handle the embeddings you set, or adjust accordingly.
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Clean up any existing Chroma DB directory to avoid conflicts
# This will create a fresh vector database each time you run the notebook.
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_DIR
)

print("✅ Local Vector DB created at:", DB_DIR)
print("✅ Total chunks indexed:", len(chunks))

In [ ]:
# Check if the database is actually linked
try:
    count = vectordb._collection.count()
    print(f"✅ Success! Found {count} chunks in your local database.")
except NameError:
    print("❌ Error: 'vectordb' is not defined. Run the 'Load Only' cell first!")

In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

DB_DIR = "./chroma_db"
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Connect to the existing data
vectordb = Chroma(
    persist_directory=DB_DIR, 
    embedding_function=embeddings
)

# Final Verification
try:
    count = vectordb._collection.count()
    print(f"✅ SUCCESS: Linked to {count} chunks in your local database!")
except Exception as e:
    print(f"❌ STILL FAILED: {e}")

### The Game Codex (Main Logic)
###### For CLI Only. Use app.py for Streamlit UI.

In [ ]:
from langchain_ollama import OllamaLLM
from collections import Counter
from rapidfuzz import process, fuzz
import re

# 1. Initialize chat model
llm = OllamaLLM(model="phi3:mini", temperature=0)

# 2. Session State
chat_history = []
current_game = None
HOUSE_RULES_RAW = {}
HOUSE_ASKED = set()

VALID_GAMES = {"uno", "monopoly", "catan", "monopoly_deal", "root", "betrayal", "sushi_go", "pandemic", "ticket_to_ride", "risk"}

DISPLAY_NAMES = {
    "uno": "UNO", "monopoly": "Monopoly", "catan": "Catan", 
    "monopoly_deal": "Monopoly Deal", "root": "Root", 
    "betrayal": "Betrayal at House on the Hill", "sushi_go": "Sushi Go!", 
    "pandemic": "Pandemic", "ticket_to_ride": "Ticket to Ride", "risk": "Risk"
}

# --- Utilities ---
def normalize_text(s: str) -> str:
    return re.sub(r"[\s_]+", " ", re.sub(r"[^a-z0-9\s_]", " ", s.lower().strip())).strip()

def best_game_match(user_text: str, cutoff: int = 70):
    ut = normalize_text(user_text)
    choices = {gid: gid for gid in VALID_GAMES}
    for gid, dn in DISPLAY_NAMES.items():
        choices[normalize_text(dn)] = gid
    match = process.extractOne(ut, list(choices.keys()), scorer=fuzz.WRatio)
    if match and match[1] >= cutoff:
        return choices[match[0]], match[1]
    return None, match[1] if match else 0

def retrieve_docs(query, game=None, k=6):
    filter_dict = {"game": game} if game else None
    return vectordb.similarity_search(query, k=k, filter=filter_dict)

def build_answer(user_question: str, game_id: str, docs):
    context = "\n\n".join([d.page_content for d in docs])
    prompt = f"You are The Game Codex. Use this context to answer: {context}\n\nQuestion: {user_question}"
    return llm.invoke(prompt)

# --- Main Loop ---
print("🎲 The Game Codex is ready!")
print("Commands: Type '/switch' to change games or '/exit' to quit.\n")

while True:
    user_input = input("You: ").strip()
    if not user_input: continue
    
    # --- COMMAND CHECK (TOP PRIORITY) ---
    clean_input = user_input.lower().strip()
    
    if clean_input in ["/exit", "exit", "quit"]:
        print("Codex: Goodbye!")
        break
    
    if clean_input == "/switch":
        current_game = None
        chat_history = []
        print("\n🔄 SYSTEM RESET: Game cleared. Which game are you playing now?")
        continue

    # 1. Game Selection Phase
    if current_game is None:
        g, score = best_game_match(user_input)
        if g:
            current_game = g
            print(f"Codex: Locked to {DISPLAY_NAMES[g]}. Ask your first question!")
        else:
            print(f"Codex: I don't recognize '{user_input}'. Please type a game like Uno, Risk, or Monopoly.")
        continue

    # 2. Rules Chat Phase
    print(f"DEBUG: Searching {current_game} rules...")
    docs = retrieve_docs(user_input, game=current_game)
    
    print("DEBUG: Thinking...")
    answer = build_answer(user_input, current_game, docs)
    
    print(f"\nCodex: {answer}\n")